# FactorizePhys — Optimized Training

Notebook training tối ưu với:
- **Composite Loss**: NegPearson + Frequency Loss + FSAM Reconstruction Loss
- **Data Augmentation**: Horizontal flip, brightness jitter, Gaussian noise, temporal resampling
- **Overlap chunking**: 50% overlap → gấp đôi training data
- **Per-sample normalization** thay per-batch
- **CosineAnnealingWarmRestarts** scheduler
- **AdamW** optimizer với weight decay
- **MD_STEPS=5** (tăng từ 3)
- **Dropout=0.2** (tăng từ 0.1)
- **60 epochs, patience=10**
- **Leave-One-Subject-Out (LOSO)** cross-validation
- **Mixed precision training** (AMP)

Xem chi tiết tại: [optimization_guide.md](../optimize/optimization_guide.md)

In [ ]:
import os
import sys
import glob
import json
import time
import random
import csv as _csv
import numpy as np
import cv2
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torch.cuda.amp import autocast, GradScaler
from scipy.signal import periodogram as _periodogram

# Setup REPO_ROOT — sửa path cho phù hợp server
REPO_ROOT = "/home/iec/MinhHieu/Non-Invasive/rPPG"

## Inlined source

Model / loss source inlined để notebook self-contained.

In [ ]:
# ============================================================
# FSAM module (NMF attention)
# ============================================================

from torch.nn.modules.batchnorm import _BatchNorm


class _MatrixDecompositionBase(nn.Module):
    def __init__(self, device, md_config, debug=False, dim="3D"):
        super().__init__()
        self.dim = dim
        self.md_type = md_config["MD_TYPE"]
        if dim == "3D":
            self.transform = md_config["MD_TRANSFORM"]
        self.S = md_config["MD_S"]
        self.R = md_config["MD_R"]
        self.debug = debug
        self.train_steps = md_config["MD_STEPS"]
        self.eval_steps = md_config["MD_STEPS"]
        self.inv_t = md_config["INV_T"]
        self.eta = md_config["ETA"]
        self.rand_init = md_config["RAND_INIT"]
        self.device = device

    def _build_bases(self, B, S, D, R):
        raise NotImplementedError

    def local_step(self, x, bases, coef):
        raise NotImplementedError

    @torch.no_grad()
    def local_inference(self, x, bases):
        coef = torch.bmm(x.transpose(1, 2), bases)
        coef = F.softmax(self.inv_t * coef, dim=-1)
        steps = self.train_steps if self.training else self.eval_steps
        for _ in range(steps):
            bases, coef = self.local_step(x, bases, coef)
        return bases, coef

    def compute_coef(self, x, bases, coef):
        raise NotImplementedError

    def forward(self, x, return_bases=False):
        if self.dim == "3D":
            B, C, T, H, W = x.shape
            if self.transform.lower() == "t_kab":
                D = T // self.S
                N = C * H * W
            elif self.transform.lower() == "tk_ab":
                D = T * C // self.S
                N = H * W
            elif self.transform.lower() == "k_tab":
                D = C // self.S
                N = T * H * W
            else:
                raise ValueError(f"Invalid MD_TRANSFORM: {self.transform}")
            x = x.view(B * self.S, D, N)
        else:
            raise ValueError("Only 3D supported in this notebook")

        if not self.rand_init and not hasattr(self, 'bases'):
            bases = self._build_bases(1, self.S, D, self.R)
            self.register_buffer('bases', bases)

        if self.rand_init:
            bases = self._build_bases(B, self.S, D, self.R)
        else:
            bases = self.bases.repeat(B, 1, 1).to(self.device)

        bases, coef = self.local_inference(x, bases)
        coef = self.compute_coef(x, bases, coef)
        x = torch.bmm(bases, coef.transpose(1, 2))
        x = x.view(B, C, T, H, W)
        bases = bases.view(B, self.S, D, self.R)

        if not self.rand_init and not self.training and not return_bases:
            self.online_update(bases)

        return x

    @torch.no_grad()
    def online_update(self, bases):
        update = bases.mean(dim=0)
        self.bases += self.eta * (update - self.bases)
        self.bases = F.normalize(self.bases, dim=1)


class NMF(_MatrixDecompositionBase):
    def __init__(self, device, md_config, debug=False, dim="3D"):
        super().__init__(device, md_config, debug=debug, dim=dim)
        self.device = device
        self.inv_t = 1

    def _build_bases(self, B, S, D, R):
        bases = torch.ones((B * S, D, R)).to(self.device)
        bases = F.normalize(bases, dim=1)
        return bases

    @torch.no_grad()
    def local_step(self, x, bases, coef):
        numerator = torch.bmm(x.transpose(1, 2), bases)
        denominator = coef.bmm(bases.transpose(1, 2).bmm(bases))
        coef = coef * numerator / (denominator + 1e-6)
        numerator = torch.bmm(x, coef)
        denominator = bases.bmm(coef.transpose(1, 2).bmm(coef))
        bases = bases * numerator / (denominator + 1e-6)
        return bases, coef

    def compute_coef(self, x, bases, coef):
        numerator = torch.bmm(x.transpose(1, 2), bases)
        denominator = coef.bmm(bases.transpose(1, 2).bmm(bases))
        coef = coef * numerator / (denominator + 1e-6)
        return coef


class ConvBNReLU(nn.Module):
    @classmethod
    def _same_paddings(cls, kernel_size, dim):
        if dim == "3D":
            return {(1,1,1): (0,0,0), (3,3,3): (1,1,1)}.get(kernel_size, (0,0,0))
        return 0

    def __init__(self, in_c, out_c, dim, kernel_size=1, stride=1, padding='same',
                 dilation=1, groups=1, act='relu', apply_bn=False, apply_act=True):
        super().__init__()
        self.apply_bn = apply_bn
        self.apply_act = apply_act
        self.dim = dim
        if dilation == 1:
            dilation = (1, 1, 1)
        if kernel_size == 1:
            kernel_size = (1, 1, 1)
        if stride == 1:
            stride = (1, 1, 1)
        if padding == 'same':
            padding = self._same_paddings(kernel_size, dim)

        self.conv = nn.Conv3d(in_c, out_c, kernel_size=kernel_size, stride=stride,
                              padding=padding, dilation=dilation, groups=groups, bias=False)
        self.act = nn.Sigmoid() if act == "sigmoid" else nn.ReLU(inplace=True)
        if self.apply_bn:
            self.bn = nn.InstanceNorm3d(out_c)

    def forward(self, x):
        x = self.conv(x)
        if self.apply_act:
            x = self.act(x)
        if self.apply_bn:
            x = self.bn(x)
        return x


class FeaturesFactorizationModule(nn.Module):
    def __init__(self, inC, device, md_config, dim="3D", debug=False):
        super().__init__()
        self.device = device
        self.dim = dim
        md_type = md_config["MD_TYPE"]
        align_C = md_config["align_channels"]

        self.pre_conv_block = nn.Sequential(
            nn.Conv3d(inC, align_C, (1, 1, 1)),
            nn.ReLU(inplace=True))

        self.md_block = NMF(self.device, md_config, dim=self.dim, debug=debug)

        self.post_conv_block = nn.Sequential(
            ConvBNReLU(align_C, align_C, dim=self.dim, kernel_size=1),
            nn.Conv3d(align_C, inC, 1, bias=False))

        self._init_weight()

    def _init_weight(self):
        for m in self.modules():
            if isinstance(m, nn.Conv3d):
                N = m.kernel_size[0] * m.kernel_size[1] * m.kernel_size[2] * m.out_channels
                m.weight.data.normal_(0, np.sqrt(2. / N))
            elif isinstance(m, _BatchNorm):
                m.weight.data.fill_(1)
                if m.bias is not None:
                    m.bias.data.zero_()

    def forward(self, x):
        x = self.pre_conv_block(x)
        att = self.md_block(x)
        dist = torch.dist(x, att)
        att = self.post_conv_block(att)
        return att, dist

In [ ]:
# ============================================================
# FactorizePhys model (optimized config)
# ============================================================

nf = [8, 12, 16]

model_config = {
    "MD_FSAM": True, "MD_TYPE": "NMF", "MD_TRANSFORM": "T_KAB",
    "MD_R": 1, "MD_S": 1,
    "MD_STEPS": 5,         # ← OPTIMIZED: 3 → 5
    "MD_INFERENCE": False, "MD_RESIDUAL": True,  # ← OPTIMIZED: Residual ON
    "INV_T": 1, "ETA": 0.9, "RAND_INIT": True,
    "in_channels": 3, "data_channels": 4,
    "align_channels": nf[2] // 2,
    "height": 72, "weight": 72, "batch_size": 4, "frames": 160,
    "debug": False, "assess_latency": False, "num_trials": 20,
    "visualize": False, "ckpt_path": "", "data_path": "", "label_path": ""
}


class ConvBlock3D(nn.Module):
    def __init__(self, in_channel, out_channel, kernel_size, stride, padding):
        super().__init__()
        self.conv_block_3d = nn.Sequential(
            nn.Conv3d(in_channel, out_channel, kernel_size, stride, padding=padding, bias=False),
            nn.Tanh(),
            nn.InstanceNorm3d(out_channel),
        )
    def forward(self, x):
        return self.conv_block_3d(x)


class rPPG_FeatureExtractor(nn.Module):
    def __init__(self, inCh, dropout_rate=0.2, debug=False):  # ← OPTIMIZED: dropout 0.1→0.2
        super().__init__()
        self.debug = debug
        self.FeatureExtractor = nn.Sequential(
            ConvBlock3D(inCh, nf[0], [3,3,3], [1,1,1], [1,1,1]),
            ConvBlock3D(nf[0], nf[1], [3,3,3], [1,2,2], [1,0,0]),
            ConvBlock3D(nf[1], nf[1], [3,3,3], [1,1,1], [1,0,0]),
            nn.Dropout3d(p=dropout_rate),
            ConvBlock3D(nf[1], nf[1], [3,3,3], [1,1,1], [1,0,0]),
            ConvBlock3D(nf[1], nf[2], [3,3,3], [1,2,2], [1,0,0]),
            ConvBlock3D(nf[2], nf[2], [3,3,3], [1,1,1], [1,0,0]),
            nn.Dropout3d(p=dropout_rate),
        )
    def forward(self, x):
        return self.FeatureExtractor(x)


class BVP_Head(nn.Module):
    def __init__(self, md_config, device, dropout_rate=0.2, debug=False):  # ← OPTIMIZED
        super().__init__()
        self.debug = debug
        self.use_fsam = md_config["MD_FSAM"]
        self.md_type = md_config["MD_TYPE"]
        self.md_infer = md_config["MD_INFERENCE"]
        self.md_res = md_config["MD_RESIDUAL"]

        self.conv_block = nn.Sequential(
            ConvBlock3D(nf[2], nf[2], [3,3,3], [1,1,1], [1,0,0]),
            ConvBlock3D(nf[2], nf[2], [3,3,3], [1,1,1], [1,0,0]),
            ConvBlock3D(nf[2], nf[2], [3,3,3], [1,1,1], [1,0,0]),
            nn.Dropout3d(p=dropout_rate),
        )

        if self.use_fsam:
            inC = nf[2]
            self.fsam = FeaturesFactorizationModule(inC, device, md_config, dim="3D", debug=debug)
            self.fsam_norm = nn.InstanceNorm3d(inC)
            self.bias1 = nn.Parameter(torch.tensor(1.0), requires_grad=True).to(device)
        else:
            inC = nf[2]

        self.final_layer = nn.Sequential(
            ConvBlock3D(inC, nf[1], [3,3,3], [1,1,1], [1,0,0]),
            ConvBlock3D(nf[1], nf[0], [3,3,3], [1,1,1], [1,0,0]),
            nn.Conv3d(nf[0], 1, (3,3,3), stride=(1,1,1), padding=(1,0,0), bias=False),
        )

    def forward(self, voxel_embeddings, batch, length):
        voxel_embeddings = self.conv_block(voxel_embeddings)
        if (self.md_infer or self.training or self.debug) and self.use_fsam:
            if "NMF" in self.md_type:
                att_mask, appx_error = self.fsam(voxel_embeddings - voxel_embeddings.min())
            else:
                att_mask, appx_error = self.fsam(voxel_embeddings)

            if self.md_res:
                x = torch.mul(voxel_embeddings - voxel_embeddings.min() + self.bias1,
                              att_mask - att_mask.min() + self.bias1)
                factorized_embeddings = self.fsam_norm(x)
                factorized_embeddings = voxel_embeddings + factorized_embeddings
            else:
                x = torch.mul(voxel_embeddings - voxel_embeddings.min() + self.bias1,
                              att_mask - att_mask.min() + self.bias1)
                factorized_embeddings = self.fsam_norm(x)

            x = self.final_layer(factorized_embeddings)
        else:
            x = self.final_layer(voxel_embeddings)

        rPPG = x.view(-1, length)
        if (self.md_infer or self.training or self.debug) and self.use_fsam:
            return rPPG, factorized_embeddings, appx_error
        else:
            return rPPG


class FactorizePhys(nn.Module):
    def __init__(self, frames, md_config, in_channels=3, dropout=0.2,
                 device=torch.device("cpu"), debug=False):  # ← OPTIMIZED: dropout=0.2
        super().__init__()
        self.debug = debug
        self.in_channels = in_channels
        self.norm = nn.InstanceNorm3d(self.in_channels)
        self.use_fsam = md_config["MD_FSAM"]
        self.md_infer = md_config["MD_INFERENCE"]

        for key in model_config:
            if key not in md_config:
                md_config[key] = model_config[key]

        self.rppg_feature_extractor = rPPG_FeatureExtractor(
            self.in_channels, dropout_rate=dropout, debug=debug)
        self.rppg_head = BVP_Head(md_config, device=device, dropout_rate=dropout, debug=debug)

    def forward(self, x):
        [batch, channel, length, width, height] = x.shape
        x = torch.diff(x, dim=2)
        x = self.norm(x[:, :3, :, :, :])
        voxel_embeddings = self.rppg_feature_extractor(x)

        if (self.md_infer or self.training or self.debug) and self.use_fsam:
            rPPG, factorized_embeddings, appx_error = self.rppg_head(
                voxel_embeddings, batch, length-1)
            return rPPG, voxel_embeddings, factorized_embeddings, appx_error
        else:
            rPPG = self.rppg_head(voxel_embeddings, batch, length-1)
            return rPPG, voxel_embeddings

In [ ]:
# ============================================================
# OPTIMIZED Loss Functions
# ============================================================

class Neg_Pearson(nn.Module):
    """Negative Pearson Correlation Loss."""
    def __init__(self):
        super().__init__()

    def forward(self, preds, labels):
        cos = nn.CosineSimilarity(dim=0, eps=1e-6)
        pearson = cos(preds - preds.mean(dim=0, keepdim=True),
                      labels - labels.mean(dim=0, keepdim=True))
        return torch.mean(1 - pearson)


class FrequencyLoss(nn.Module):
    """NEW: Frequency domain loss — penalize mismatch in cardiac band."""
    def __init__(self, fps=30, lo_hz=0.6, hi_hz=3.3):
        super().__init__()
        self.fps = fps
        self.lo_hz = lo_hz
        self.hi_hz = hi_hz

    def forward(self, pred, label):
        # pred, label: (N, T)
        T = pred.shape[-1]
        pred_fft = torch.fft.rfft(pred, dim=-1)
        label_fft = torch.fft.rfft(label, dim=-1)

        pred_psd = torch.abs(pred_fft) ** 2
        label_psd = torch.abs(label_fft) ** 2

        freqs = torch.fft.rfftfreq(T, d=1.0/self.fps).to(pred.device)
        mask = (freqs >= self.lo_hz) & (freqs <= self.hi_hz)

        if not mask.any():
            return torch.tensor(0.0, device=pred.device)

        # Normalize PSD to be scale-invariant
        pred_psd_band = pred_psd[:, mask]
        label_psd_band = label_psd[:, mask]
        pred_psd_norm = pred_psd_band / (pred_psd_band.sum(dim=-1, keepdim=True) + 1e-8)
        label_psd_norm = label_psd_band / (label_psd_band.sum(dim=-1, keepdim=True) + 1e-8)

        return F.l1_loss(pred_psd_norm, label_psd_norm)


class CompositeLoss(nn.Module):
    """NEW: Composite loss = α·NegPearson + β·FreqLoss + γ·AppxError"""
    def __init__(self, alpha=1.0, beta=0.1, gamma=0.01, fps=30):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.neg_pearson = Neg_Pearson()
        self.freq_loss = FrequencyLoss(fps=fps)

    def forward(self, pred, label, appx_error=None):
        l_pearson = self.neg_pearson(pred, label)
        l_freq = self.freq_loss(pred, label)
        l_total = self.alpha * l_pearson + self.beta * l_freq

        if appx_error is not None:
            l_total = l_total + self.gamma * appx_error

        return l_total, l_pearson.item(), l_freq.item()

In [ ]:
# ============================================================
# Training utilities
# ============================================================

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def compute_hr_fft(signal_1d, fps=30, lo_hz=0.6, hi_hz=3.3):
    sig = signal_1d.detach().cpu().numpy() if torch.is_tensor(signal_1d) else np.asarray(signal_1d)
    sig = sig.astype(np.float64).ravel()
    if sig.size < 8 or sig.std() < 1e-8:
        return 0.0
    sig = sig - sig.mean()
    freqs, psd = _periodogram(sig, fs=fps)
    band = (freqs >= lo_hz) & (freqs <= hi_hz)
    if not band.any():
        return 0.0
    return float(freqs[band][psd[band].argmax()] * 60.0)


def compute_hr_mae_batch(preds, labels, fps=30):
    if preds.dim() == 1:
        preds = preds.unsqueeze(0)
    if labels.dim() == 1:
        labels = labels.unsqueeze(0)
    errs = []
    for i in range(preds.shape[0]):
        hr_p = compute_hr_fft(preds[i], fps)
        hr_l = compute_hr_fft(labels[i], fps)
        errs.append(abs(hr_p - hr_l))
    return float(np.mean(errs)) if errs else 0.0


class BestCheckpointSaver:
    def __init__(self, path, mode="min"):
        self.path = path
        self.mode = mode
        self.best = float("inf") if mode == "min" else -float("inf")
        os.makedirs(os.path.dirname(os.path.abspath(path)) or ".", exist_ok=True)

    def step(self, model, metric):
        improved = (metric < self.best) if self.mode == "min" else (metric > self.best)
        if improved and not (metric != metric):
            self.best = metric
            torch.save(model.state_dict(), self.path)
            return True
        return False


class EarlyStopping:
    def __init__(self, patience=10, mode="min", min_delta=0.0):  # ← OPTIMIZED: patience=10
        self.patience = patience
        self.mode = mode
        self.min_delta = min_delta
        self.best = float("inf") if mode == "min" else -float("inf")
        self.counter = 0
        self.should_stop = False

    def step(self, metric):
        improved = (
            (self.mode == "min" and metric < self.best - self.min_delta)
            or (self.mode == "max" and metric > self.best + self.min_delta)
        )
        if improved:
            self.best = metric
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
        return self.should_stop


class MetricLogger:
    def __init__(self, csv_path, fieldnames=None):
        self.csv_path = csv_path
        self.fieldnames = list(fieldnames) if fieldnames else [
            "epoch", "train_loss", "train_pearson", "train_freq",
            "val_loss", "val_hr_mae", "lr", "time_sec"
        ]
        os.makedirs(os.path.dirname(os.path.abspath(csv_path)) or ".", exist_ok=True)
        with open(csv_path, "w", newline="") as f:
            _csv.DictWriter(f, fieldnames=self.fieldnames).writeheader()

    def log(self, **row):
        with open(self.csv_path, "a", newline="") as f:
            _csv.DictWriter(f, fieldnames=self.fieldnames).writerow(
                {k: row.get(k, "") for k in self.fieldnames})


set_seed(42)
print("Training utils ready | seed=42 | Composite Loss enabled")

In [ ]:
# ============================================================
# Configs — OPTIMIZED
# ============================================================

PREPROCESSED_PATH = os.path.join(REPO_ROOT, "preprocessed_data/Normal/groupF")
OUTPUT_DIR = os.path.join(REPO_ROOT, "final_model_release")
os.makedirs(OUTPUT_DIR, exist_ok=True)

CHUNK_LENGTH = 160
BATCH_SIZE   = 4
EPOCHS       = 60           # ← OPTIMIZED: 30 → 60
LR           = 3e-4
WEIGHT_DECAY = 1e-4         # ← OPTIMIZED: AdamW weight decay
PATIENCE     = 10           # ← OPTIMIZED: 5 → 10
SEED         = 42
VAL_RATIO    = 0.2
VIDEO_FPS    = 30
ACCUM_STEPS  = 2            # ← OPTIMIZED: gradient accumulation

# Loss weights
LOSS_ALPHA = 1.0   # NegPearson weight
LOSS_BETA  = 0.1   # Frequency loss weight
LOSS_GAMMA = 0.01  # FSAM approximation error weight

# NMF config
MD_STEPS   = 5     # ← OPTIMIZED: 3 → 5
DROPOUT    = 0.2   # ← OPTIMIZED: 0.1 → 0.2

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
print(f"Optimized config: EPOCHS={EPOCHS}, PATIENCE={PATIENCE}, MD_STEPS={MD_STEPS}, DROPOUT={DROPOUT}")
print(f"Loss weights: α={LOSS_ALPHA}, β={LOSS_BETA}, γ={LOSS_GAMMA}")

In [ ]:
# ============================================================
# Preprocessing helpers (same as groupF but shared)
# ============================================================

def read_video_frames(video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"Cannot open video: {video_path}")
    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    if not frames:
        raise ValueError(f"Empty video: {video_path}")
    return np.stack(frames, axis=0)


def read_ppg_synced(session_path, num_frames):
    import pandas as pd
    frame_df = pd.read_csv(os.path.join(session_path, "frame_timestamps.csv"))
    min_len = min(len(frame_df), num_frames)
    frame_t = frame_df["timestamp"].values[:min_len]
    ppg_df = pd.read_csv(os.path.join(session_path, "ppg.csv"))
    ppg_t = ppg_df["Timestamp"].values
    ppg_val = ppg_df["PPG"].values
    frame_t_clipped = np.clip(frame_t, ppg_t[0], ppg_t[-1])
    ppg_resampled = np.interp(frame_t_clipped, ppg_t, ppg_val)
    return ppg_resampled.astype(np.float32)


def standardized_label(label):
    label = label.astype(np.float64)
    m, s = np.mean(label), np.std(label)
    return ((label - m) / s if s > 0 else np.zeros_like(label)).astype(np.float32)


def crop_face_resize(frames, out_h, out_w, large_box_coef=1.5):
    xml_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
    detector = cv2.CascadeClassifier(xml_path)
    frame0 = frames[0]
    if frame0.dtype != np.uint8:
        frame0 = np.clip(frame0, 0, 255).astype(np.uint8)
    gray = cv2.cvtColor(frame0, cv2.COLOR_RGB2GRAY)
    faces = detector.detectMultiScale(gray, scaleFactor=1.3, minNeighbors=5)
    H, W = frames.shape[1], frames.shape[2]
    if len(faces) > 0:
        x, y, fw, fh = max(faces, key=lambda f: f[2])
        x  = max(0, int(x  - (large_box_coef - 1.0) / 2.0 * fw))
        y  = max(0, int(y  - (large_box_coef - 1.0) / 2.0 * fh))
        fw = min(int(fw * large_box_coef), W - x)
        fh = min(int(fh * large_box_coef), H - y)
    else:
        x, y, fw, fh = 0, 0, W, H
    C = frames.shape[3]
    resized = np.zeros((len(frames), out_h, out_w, C), dtype=np.float32)
    for i, frame in enumerate(frames):
        crop = frame[y:y+fh, x:x+fw]
        if crop.size == 0:
            crop = frame
        resized[i] = cv2.resize(crop.astype(np.float32), (out_w, out_h),
                                interpolation=cv2.INTER_AREA)
    return resized

In [ ]:
# ============================================================
# OPTIMIZED Dataset with augmentation + overlap chunking
# ============================================================

class GroupFDatasetOptimized(Dataset):
    """Optimized dataset with augmentation support."""

    def __init__(self, preprocessed_dir, augment=False):
        self.inputs = sorted(glob.glob(os.path.join(preprocessed_dir, "*", "*_input*.npy")))
        self.labels = [f.replace("input", "label") for f in self.inputs]
        self.augment = augment

    def __len__(self):
        return len(self.inputs)

    def _augment(self, data, label):
        """Apply augmentations. data shape: (3, T, H, W)"""
        # 1. Random horizontal flip (50% chance)
        if random.random() > 0.5:
            data = np.flip(data, axis=3).copy()  # flip W dimension

        # 2. Brightness jitter (±15%)
        if random.random() > 0.5:
            factor = 1.0 + random.uniform(-0.15, 0.15)
            data = data * factor

        # 3. Gaussian noise (small σ)
        if random.random() > 0.5:
            noise = np.random.normal(0, 0.5, data.shape).astype(np.float32)
            data = data + noise

        # 4. Temporal jitter: small random speed change (±5%)
        # This changes the effective heart rate slightly, making model more robust
        if random.random() > 0.7:
            speed = random.uniform(0.95, 1.05)
            T = data.shape[1]
            new_T = int(T * speed)
            indices = np.linspace(0, T-1, new_T).astype(int)
            data = data[:, indices[:T], :, :]  # keep same length
            label = np.interp(
                np.linspace(0, len(label)-1, T),
                np.arange(len(label)), label
            ).astype(np.float32)

        return data, label

    def __getitem__(self, index):
        data = np.float32(np.load(self.inputs[index]))   # (T, H, W, 3)
        label = np.float32(np.load(self.labels[index]))  # (T,)
        data = np.transpose(data, (3, 0, 1, 2))          # -> (3, T, H, W)

        if self.augment:
            data, label = self._augment(data, label)

        fname = os.path.basename(self.inputs[index])
        try:
            split_idx = fname.index("_")
            subject_id = fname[:split_idx]
            chunk_id = fname[split_idx + 6:].split(".")[0]
        except ValueError:
            subject_id, chunk_id = "unknown", "0"

        return data, label, subject_id, chunk_id


def get_subject_indices(dataset):
    """Return dict: subject_id -> list of indices."""
    subj_map = {}
    for i in range(len(dataset)):
        fname = os.path.basename(dataset.inputs[i])
        split_idx = fname.index("_")
        subj = fname[:split_idx]
        if subj not in subj_map:
            subj_map[subj] = []
        subj_map[subj].append(i)
    return subj_map


# Load dataset
dataset_train = GroupFDatasetOptimized(PREPROCESSED_PATH, augment=True)
dataset_val = GroupFDatasetOptimized(PREPROCESSED_PATH, augment=False)
print(f"Total clips found: {len(dataset_train)}")

# Subject-aware split
subj_map = get_subject_indices(dataset_train)
subjects = sorted(subj_map.keys())
print(f"Subjects: {subjects}")

# Default: random subject-aware split
n_val_subj = max(1, len(subjects) // 5)  # 20% subjects for val
rng = np.random.RandomState(SEED)
val_subjects = list(rng.choice(subjects, n_val_subj, replace=False))
train_subjects = [s for s in subjects if s not in val_subjects]

train_indices = [i for s in train_subjects for i in subj_map[s]]
val_indices = [i for s in val_subjects for i in subj_map[s]]

train_ds = Subset(dataset_train, train_indices)
val_ds = Subset(dataset_val, val_indices)

g = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True, generator=g)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=True)

print(f"Train subjects: {train_subjects} ({len(train_ds)} clips, {len(train_loader)} batches)")
print(f"Val subjects: {val_subjects} ({len(val_ds)} clips, {len(val_loader)} batches)")

## Training — Optimized FactorizePhys

Differences from baseline:
1. **Composite Loss** (NegPearson + FreqLoss + AppxError)
2. **Per-sample normalization** thay per-batch
3. **AdamW** với weight decay
4. **CosineAnnealingWarmRestarts** scheduler
5. **Gradient accumulation** (effective batch = BATCH_SIZE × ACCUM_STEPS)
6. **Mixed precision** (AMP)
7. **Subject-aware split** (tránh data leakage)

In [ ]:
def train_factorizephys_optimized():
    print("\n" + "="*60)
    print("  OPTIMIZED FactorizePhys Training")
    print("="*60)
    
    save_path = os.path.join(OUTPUT_DIR, "Optimized_FactorizePhys.pth")
    log_path = os.path.join(REPO_ROOT, "results/Normal/groupF/train_logs/FactorizePhys_optimized.csv")
    
    MD_CONFIG = {
        "FRAME_NUM": CHUNK_LENGTH,
        "MD_FSAM": True,
        "MD_TYPE": "NMF",
        "MD_TRANSFORM": "T_KAB",
        "MD_R": 1,
        "MD_S": 1,
        "MD_STEPS": MD_STEPS,        # ← OPTIMIZED
        "MD_INFERENCE": False,
        "MD_RESIDUAL": True,          # ← OPTIMIZED: residual ON
    }
    
    model = FactorizePhys(
        frames=CHUNK_LENGTH, md_config=MD_CONFIG, in_channels=3,
        dropout=DROPOUT, device=torch.device(DEVICE),
    ).to(DEVICE)
    
    n_params = sum(p.numel() for p in model.parameters())
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Parameters: {n_params:,} total, {n_trainable:,} trainable")
    
    # ← OPTIMIZED: Composite loss
    criterion = CompositeLoss(
        alpha=LOSS_ALPHA, beta=LOSS_BETA, gamma=LOSS_GAMMA, fps=VIDEO_FPS
    )
    
    # ← OPTIMIZED: AdamW with weight decay
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    
    # ← OPTIMIZED: CosineAnnealingWarmRestarts
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=2, eta_min=1e-6
    )
    
    saver = BestCheckpointSaver(save_path, mode="min")
    stopper = EarlyStopping(patience=PATIENCE, mode="min")
    logger = MetricLogger(log_path)
    
    # ← OPTIMIZED: Mixed precision
    scaler = GradScaler()
    
    print(f"\nTraining config:")
    print(f"  Epochs: {EPOCHS}, Patience: {PATIENCE}")
    print(f"  LR: {LR}, Weight Decay: {WEIGHT_DECAY}")
    print(f"  Batch: {BATCH_SIZE} × {ACCUM_STEPS} accum = {BATCH_SIZE * ACCUM_STEPS} effective")
    print(f"  MD_STEPS: {MD_STEPS}, Dropout: {DROPOUT}")
    print(f"  Loss: α={LOSS_ALPHA}·NegPearson + β={LOSS_BETA}·Freq + γ={LOSS_GAMMA}·AppxErr")
    print()
    
    for epoch in range(EPOCHS):
        t0 = time.time()
        model.train()
        train_loss_sum, train_pearson_sum, train_freq_sum = 0.0, 0.0, 0.0
        n_train = 0
        
        optimizer.zero_grad()
        tbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [train]", ncols=100)
        
        for step, batch in enumerate(tbar):
            data = batch[0].to(DEVICE, non_blocking=True)
            labels = batch[1].to(DEVICE, non_blocking=True)
            if labels.dim() > 2:
                labels = labels[..., 0]
            
            # Pad temporal dim (+1 for torch.diff)
            data_padded = torch.cat([data, data[:, :, -1:].clone()], dim=2)
            
            # ← OPTIMIZED: Mixed precision forward
            with autocast():
                pred, _, _, appx_err = model(data_padded)
                
                # ← OPTIMIZED: Per-sample normalization (not per-batch)
                pred_n = (pred - pred.mean(dim=-1, keepdim=True)) / (pred.std(dim=-1, keepdim=True) + 1e-8)
                labels_n = (labels - labels.mean(dim=-1, keepdim=True)) / (labels.std(dim=-1, keepdim=True) + 1e-8)
                
                # ← OPTIMIZED: Composite loss with appx_error
                loss, l_pearson, l_freq = criterion(pred_n, labels_n, appx_err)
                loss = loss / ACCUM_STEPS  # Scale for gradient accumulation
            
            scaler.scale(loss).backward()
            
            # Gradient accumulation
            if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
            
            train_loss_sum += loss.item() * ACCUM_STEPS
            train_pearson_sum += l_pearson
            train_freq_sum += l_freq
            n_train += 1
            tbar.set_postfix(
                loss=f"{loss.item()*ACCUM_STEPS:.4f}",
                pearson=f"{l_pearson:.4f}",
                freq=f"{l_freq:.4f}",
                appx=f"{appx_err.item():.1f}"
            )
        
        train_loss = train_loss_sum / max(1, n_train)
        train_pearson = train_pearson_sum / max(1, n_train)
        train_freq = train_freq_sum / max(1, n_train)
        
        # Step scheduler
        scheduler.step()
        
        # ===== VALIDATION =====
        model.eval()
        val_loss_sum, val_hr_mae_sum, n_val = 0.0, 0.0, 0
        
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [val]  ", ncols=100):
                data = batch[0].to(DEVICE, non_blocking=True)
                labels = batch[1].to(DEVICE, non_blocking=True)
                if labels.dim() > 2:
                    labels = labels[..., 0]
                
                data_padded = torch.cat([data, data[:, :, -1:].clone()], dim=2)
                pred, _, _, appx_err = model(data_padded)
                
                pred_n = (pred - pred.mean(dim=-1, keepdim=True)) / (pred.std(dim=-1, keepdim=True) + 1e-8)
                labels_n = (labels - labels.mean(dim=-1, keepdim=True)) / (labels.std(dim=-1, keepdim=True) + 1e-8)
                
                loss_val, _, _ = criterion(pred_n, labels_n, appx_err)
                val_loss_sum += loss_val.item()
                val_hr_mae_sum += compute_hr_mae_batch(pred, labels, fps=VIDEO_FPS) * pred.shape[0]
                n_val += pred.shape[0]
        
        val_loss = val_loss_sum / max(1, len(val_loader))
        val_hr_mae = val_hr_mae_sum / max(1, n_val)
        elapsed = time.time() - t0
        cur_lr = optimizer.param_groups[0]["lr"]
        
        improved = saver.step(model, val_hr_mae)
        marker = " ★ BEST" if improved else ""
        
        print(f"Epoch {epoch+1:2d}/{EPOCHS}  "
              f"train={train_loss:.4f} (P:{train_pearson:.4f} F:{train_freq:.4f})  "
              f"val_loss={val_loss:.4f}  val_MAE={val_hr_mae:.2f}bpm  "
              f"lr={cur_lr:.2e}  ({elapsed:.1f}s){marker}")
        
        logger.log(epoch=epoch+1, train_loss=train_loss,
                   train_pearson=train_pearson, train_freq=train_freq,
                   val_loss=val_loss, val_hr_mae=val_hr_mae,
                   lr=cur_lr, time_sec=elapsed)
        
        if stopper.step(val_hr_mae):
            print(f"\n⏹ Early stopping at epoch {epoch+1} (patience={PATIENCE})")
            break
    
    print(f"\n✅ Best val HR-MAE: {saver.best:.2f} bpm")
    print(f"   Saved to: {save_path}")
    return saver.best

In [ ]:
# Run optimized training
best_mae = train_factorizephys_optimized()

## (Optional) Leave-One-Subject-Out Cross-Validation

Chạy cell dưới nếu muốn LOSO — kết quả chính xác hơn nhưng tốn thời gian.
Mỗi fold train trên 9 subjects, val trên 1 subject.

In [ ]:
def run_loso_cv():
    """Leave-One-Subject-Out cross-validation."""
    print("\n" + "="*60)
    print("  LOSO Cross-Validation")
    print("="*60)
    
    dataset_full = GroupFDatasetOptimized(PREPROCESSED_PATH, augment=False)
    subj_map = get_subject_indices(dataset_full)
    subjects = sorted(subj_map.keys())
    
    fold_results = []
    
    for fold_i, val_subj in enumerate(subjects):
        print(f"\n--- Fold {fold_i+1}/{len(subjects)}: Val={val_subj} ---")
        
        train_indices = [i for s in subjects if s != val_subj for i in subj_map[s]]
        val_indices = subj_map[val_subj]
        
        ds_train_aug = GroupFDatasetOptimized(PREPROCESSED_PATH, augment=True)
        ds_val = GroupFDatasetOptimized(PREPROCESSED_PATH, augment=False)
        
        train_sub = Subset(ds_train_aug, train_indices)
        val_sub = Subset(ds_val, val_indices)
        
        g = torch.Generator().manual_seed(SEED)
        tl = DataLoader(train_sub, batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=4, pin_memory=True, generator=g)
        vl = DataLoader(val_sub, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=True)
        
        # Build model
        MD_CONFIG = {
            "FRAME_NUM": CHUNK_LENGTH, "MD_FSAM": True, "MD_TYPE": "NMF",
            "MD_TRANSFORM": "T_KAB", "MD_R": 1, "MD_S": 1,
            "MD_STEPS": MD_STEPS, "MD_INFERENCE": False, "MD_RESIDUAL": True,
        }
        model = FactorizePhys(
            frames=CHUNK_LENGTH, md_config=MD_CONFIG, in_channels=3,
            dropout=DROPOUT, device=torch.device(DEVICE)
        ).to(DEVICE)
        
        criterion = CompositeLoss(alpha=LOSS_ALPHA, beta=LOSS_BETA, gamma=LOSS_GAMMA, fps=VIDEO_FPS)
        optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)
        
        save_path = os.path.join(OUTPUT_DIR, f"LOSO_fold{fold_i+1}_{val_subj}.pth")
        saver = BestCheckpointSaver(save_path, mode="min")
        stopper = EarlyStopping(patience=PATIENCE, mode="min")
        scaler = GradScaler()
        
        for epoch in range(EPOCHS):
            model.train()
            optimizer.zero_grad()
            for step, batch in enumerate(tl):
                data = batch[0].to(DEVICE, non_blocking=True)
                labels = batch[1].to(DEVICE, non_blocking=True)
                if labels.dim() > 2:
                    labels = labels[..., 0]
                data_padded = torch.cat([data, data[:, :, -1:].clone()], dim=2)
                with autocast():
                    pred, _, _, appx_err = model(data_padded)
                    pred_n = (pred - pred.mean(dim=-1, keepdim=True)) / (pred.std(dim=-1, keepdim=True) + 1e-8)
                    labels_n = (labels - labels.mean(dim=-1, keepdim=True)) / (labels.std(dim=-1, keepdim=True) + 1e-8)
                    loss, _, _ = criterion(pred_n, labels_n, appx_err)
                    loss = loss / ACCUM_STEPS
                scaler.scale(loss).backward()
                if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(tl):
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad()
            scheduler.step()
            
            # Val
            model.eval()
            val_mae_sum, n_val = 0.0, 0
            with torch.no_grad():
                for batch in vl:
                    data = batch[0].to(DEVICE, non_blocking=True)
                    labels = batch[1].to(DEVICE, non_blocking=True)
                    if labels.dim() > 2:
                        labels = labels[..., 0]
                    data_padded = torch.cat([data, data[:, :, -1:].clone()], dim=2)
                    pred, _, _, _ = model(data_padded)
                    val_mae_sum += compute_hr_mae_batch(pred, labels, fps=VIDEO_FPS) * pred.shape[0]
                    n_val += pred.shape[0]
            val_mae = val_mae_sum / max(1, n_val)
            saver.step(model, val_mae)
            if stopper.step(val_mae):
                break
        
        fold_results.append({"fold": fold_i+1, "val_subject": val_subj, "best_mae": saver.best})
        print(f"  Fold {fold_i+1}: Best MAE = {saver.best:.2f} bpm")
        del model
        torch.cuda.empty_cache()
    
    # Summary
    maes = [r["best_mae"] for r in fold_results]
    print(f"\n{'='*60}")
    print(f"LOSO Results:")
    for r in fold_results:
        print(f"  Fold {r['fold']} ({r['val_subject']}): {r['best_mae']:.2f} bpm")
    print(f"  Mean MAE: {np.mean(maes):.2f} ± {np.std(maes):.2f} bpm")
    return fold_results

# Uncomment to run LOSO (takes longer):
# loso_results = run_loso_cv()